# Entrenamiento del clasificador de estados de tráfico

Workflow de entrenamiento batch de VAAET ML 4.4.0. Separa el Inicio Semilla con weak supervision del reentrenamiento HITL recurrente; ambos convergen en el mismo MLP de tres estados y bundle portable.

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("VAAET repository root not found")

os.chdir(REPO_ROOT)
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(f"{REPO_ROOT}[training,visualization,database]")
else:
    install_command.extend(["-e", f"{REPO_ROOT}[training,visualization,database]"])
subprocess.check_call(install_command)

for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import vaaet

def validate_vaaet_origin(package: object, repo_root: Path, in_colab: bool) -> Path:
    package_file = getattr(package, "__file__", None)
    if not package_file:
        package_path = list(getattr(package, "__path__", ()))
        raise ImportError(
            "The 'vaaet' import resolved to a namespace package instead of the installed package. "
            f"Resolved locations: {package_path}. Re-run this setup cell."
        )
    origin = Path(package_file).resolve()
    expected_editable_root = (repo_root / "src/vaaet").resolve()
    if in_colab and repo_root.resolve() in origin.parents:
        raise ImportError(f"Colab must load the installed wheel, not repository path: {origin}")
    if not in_colab and origin.parent != expected_editable_root:
        raise ImportError(f"Local editable install has unexpected origin: {origin}")
    return origin

VAAET_PACKAGE_FILE = validate_vaaet_origin(vaaet, REPO_ROOT, IN_COLAB)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
pip_check_output = "\n".join(
    part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
)
if pip_check.returncode == 0:
    print("✅ pip check: no broken requirements found")
else:
    print("⚠️ pip check detected conflicts in the managed notebook runtime:")
    print(pip_check_output or "No diagnostic output was returned")
    print("ℹ️ Continuing because workflow imports are validated explicitly below.")

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not required"

print({name: package_version(name) for name in ("numpy", "tensorflow", "opencv-python-headless", "ultralytics-opencv-headless")})

import os
import shutil
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
import sqlalchemy
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, Input
from tensorflow.keras.models import Sequential

from vaaet.artifacts import FEATURE_SCHEMA_VERSION, MANIFEST_FILE, create_manifest
from vaaet.data.database import DatabaseProfile, get_engine, get_optional_database_settings
from vaaet.data.pipeline_runs import PipelineRunMetadata, PipelineWorkflow, pipeline_run
from vaaet.data.ingestion import DatasetPackageSource, FeedbackPolicy, PostgresBackupSource, PostgresSource, RawCsvSource, SeedDatasetPackageSource, TrainingIngestionPlan, compose_supervised_dataset, create_dataset_package, load_training_inputs
from vaaet.data.datasets import build_group_ids
from vaaet.data.timestamps import normalize_timestamp_series
from vaaet.evaluation.calibration import apply_temperature_scaling, fit_temperature, multiclass_brier_score
from vaaet.evaluation.dataset_validation import audit_training_dataset
from vaaet.evaluation.reporting import build_class_support_notes, build_classification_support_table, expected_calibration_error, expected_confusion_cost, select_validation_decision_policy, summarize_data_origin, summarize_state_balance
from vaaet.features.engineering import engineer_features
from vaaet.features.labeling import assign_stable_traffic_state
from vaaet.features.synthetic import augment_with_synthetic
from vaaet.inference.traffic_state import apply_conservative_accident_gate, classify_telemetry_dataframe
from vaaet.logging import configure_logging
from vaaet.settings import DATA_PROCESSED_DIR, DATA_RAW_DIR, DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABELING_THRESHOLDS, MODEL_DIR, MODEL_VERSION, N_MODEL_STATES, RANDOM_SEED, STATE_LABELS
from vaaet.training.balancing import BalanceStrategy, build_balance_candidates, compute_capped_balanced_weights
from vaaet.training.holdout import HumanHoldoutAction, HumanHoldoutConfig, resolve_human_holdout
from vaaet.training.lifecycle import ModelInputPolicy, TrainingMode, apply_model_input_policy, build_supervision_weights, build_training_lifecycle, cap_synthetic_congested_weight
from vaaet.training.partitions import build_training_partitions

configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | TensorFlow {tf.__version__} | GPU {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Package: {VAAET_PACKAGE_FILE}")
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f"✅ training workflow ready | root={REPO_ROOT} | commit={GIT_COMMIT}")

_MODEL_DIR = REPO_ROOT / MODEL_DIR
_DATA_DIR = REPO_ROOT / DATA_PROCESSED_DIR
_RAW_DIR = REPO_ROOT / DATA_RAW_DIR
for directory in (_MODEL_DIR, _DATA_DIR, _RAW_DIR):
    directory.mkdir(parents=True, exist_ok=True)


## Dos modos explícitos — Inicio Semilla y reentrenamiento HITL

`TrainingMode.SEED_BOOTSTRAP` transforma telemetría raw una sola vez y crea un paquete semilla procesado. `TrainingMode.HITL_RETRAINING` carga features ya calculadas y etiquetas humanas sin repetir ingeniería.

La carga usa un plan explícito y puede combinar varias fuentes en una misma ejecución:

1. PostgreSQL vivo mediante el perfil read-only `training`.
2. CSV raw explícito.
3. Backup `pg_dump` raw o completo.
4. Paquete contractual `vaaet-training-dataset-v1.zip`.

Cada entrada se declara como objeto tipado dentro de `RAW_SOURCES`, `SEED_SOURCES` o `FEEDBACK_SOURCES`; nunca se adivina por columnas. El feedback sólo admite validaciones humanas efectivas.

En HITL, `HUMAN_HOLDOUT_FROZEN=True` crea o reutiliza un snapshot inmutable de validation y test bajo Google Drive. `CREATE_NEW_VERSION` exige un motivo, conserva el ZIP anterior y actualiza `current.json`; los grupos congelados nunca ingresan en train.

After loading, **Cell 2b** appends synthetic Accident and Congestion stress sequences (200 records total). The historical period has no human-confirmed Accident and only limited proxy support for Congested. Synthetic incident rows never become MLP targets.

En Colab, configure `VAAET_DB_*` y `VAAET_TRAINING_DB_USER/PASSWORD` mediante Secrets. El notebook no solicita ni imprime credenciales.

In [ ]:
# Cell 1b — Data Upload (Colab only)
#
# On Colab, if no CSV cache exists, upload one of:
#   - traffic_data.backup  (pg_dump binary → processed via pg_restore)
#   - traffic_data_raw.csv (explicit RawCsvSource)
#   - vaaet-training-dataset-v1.zip (validated HITL package)
#   - vaaet-seed-bootstrap-v1.zip (processed reusable seed package)
# On local, raw/HITL files use data/raw/ and the seed package uses data/processed/.
# Select one lifecycle before uploading data.
TRAINING_MODE = TrainingMode.SEED_BOOTSTRAP
# TRAINING_MODE = TrainingMode.HITL_RETRAINING
HUMAN_HOLDOUT_FROZEN = False  # Enable only with HITL_RETRAINING and sufficient human support.
HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
# HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.CREATE_NEW_VERSION
HUMAN_HOLDOUT_UPDATE_REASON = None  # Required only for CREATE_NEW_VERSION.
# Set False when all declared sources use a live PostgreSQL server.
ENABLE_DATA_UPLOAD = True

_backup_dest = os.path.join(_RAW_DIR, "traffic_data.backup")
_csv_dest = os.path.join(_RAW_DIR, "traffic_data_raw.csv")
_package_dest = os.path.join(_RAW_DIR, "vaaet-training-dataset-v1.zip")
_seed_package_dest = os.path.join(_DATA_DIR, "vaaet-seed-bootstrap-v1.zip")
HUMAN_HOLDOUT_STORE_ROOT = _DATA_DIR / "holdouts"
if HUMAN_HOLDOUT_FROZEN and TRAINING_MODE is not TrainingMode.HITL_RETRAINING:
    raise ValueError("HUMAN_HOLDOUT_FROZEN=True is valid only for HITL_RETRAINING.")
if HUMAN_HOLDOUT_FROZEN and IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError("Frozen human holdout requires mounted Google Drive; no ephemeral fallback is allowed.") from exc
    HUMAN_HOLDOUT_STORE_ROOT = Path("/content/drive/MyDrive/vaaet-ml/data/holdouts")
    HUMAN_HOLDOUT_STORE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"🔒 Human holdout store: {HUMAN_HOLDOUT_STORE_ROOT}")

if IN_COLAB and TRAINING_MODE is TrainingMode.HITL_RETRAINING and not os.path.exists(_seed_package_dest):
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)
        _drive_seed = os.path.join("/content/drive", "MyDrive", "vaaet-ml", "data", "processed", "vaaet-seed-bootstrap-v1.zip")
        if os.path.isfile(_drive_seed):
            shutil.copy2(_drive_seed, _seed_package_dest)
            print(f"✅ Processed seed package restored from Drive: {_seed_package_dest}")
    except Exception as exc:
        print(f"ℹ️ Seed package was not restored from Drive: {exc}")

_required_local_input_exists = (
    any(os.path.exists(path) for path in (_csv_dest, _backup_dest))
    if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP
    else os.path.exists(_package_dest)
)
if IN_COLAB and ENABLE_DATA_UPLOAD and not _required_local_input_exists:
        from google.colab import files  # type: ignore[import-untyped]
        print("📤 Upload raw backup/CSV, HITL dataset package, or processed seed package:")
        uploaded = files.upload()
        if uploaded:
            import shutil as _shutil
            for fname in uploaded:
                if fname == "vaaet-seed-bootstrap-v1.zip":
                    _shutil.move(fname, _seed_package_dest)
                    print(f"✅ Processed seed package saved to {_seed_package_dest}")
                elif fname.endswith(".zip"):
                    _shutil.move(fname, _package_dest)
                    print(f"✅ Dataset package saved to {_package_dest}")
                elif fname.endswith(".csv"):
                    _shutil.move(fname, _csv_dest)
                    print(f"✅ CSV saved to {_csv_dest}; declare RawCsvSource explicitly")
                else:
                    _shutil.move(fname, _backup_dest)
                    print(f"✅ Backup saved to {_backup_dest}; declare PostgresBackupSource explicitly")
        else:
            print("⚠️ No file uploaded; declare a live PostgreSQL source or rerun this cell")
else:
    if os.path.exists(_csv_dest):
        print(f"📂 CSV cache available: {os.path.abspath(_csv_dest)}")
    elif os.path.exists(_backup_dest):
        print(f"📂 Backup available: {os.path.abspath(_backup_dest)}")
    elif os.path.exists(_package_dest):
        print(f"📂 Dataset package available: {os.path.abspath(_package_dest)}")
    elif os.path.exists(_seed_package_dest):
        print(f"📂 Processed seed package available: {os.path.abspath(_seed_package_dest)}")
    elif not ENABLE_DATA_UPLOAD:
        print("ℹ️ Upload disabled; declare PostgresSource in Cell 2")
    else:
        print("📂 No local data source present")

# A binary pg_dump needs a compatible PostgreSQL client (OS dependency).
PG_RESTORE_PATH: str | None = shutil.which("pg_restore")

if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP and os.path.exists(_backup_dest) and not os.path.exists(_csv_dest):
    if IN_COLAB:
        _pg17_binary = Path("/usr/lib/postgresql/17/bin/pg_restore")
        if not _pg17_binary.is_file():
            print("📦 Configuring the official PostgreSQL PGDG repository...")
            _apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
            _pgdg_dir = Path("/usr/share/postgresql-common/pgdg")
            _pgdg_key = _pgdg_dir / "apt.postgresql.org.asc"
            _pgdg_source = Path("/etc/apt/sources.list.d/pgdg.sources")
            try:
                subprocess.check_call(["apt-get", "update", "-qq"], env=_apt_env)
                subprocess.check_call(
                    ["apt-get", "install", "-y", "-qq", "curl", "ca-certificates"],
                    env=_apt_env,
                )
                subprocess.check_call(["install", "-d", str(_pgdg_dir)])
                subprocess.check_call(
                    [
                        "curl", "--fail", "--silent", "--show-error", "--retry", "3",
                        "--output", str(_pgdg_key),
                        "https://www.postgresql.org/media/keys/ACCC4CF8.asc",
                    ]
                )
                _os_release = {}
                for _line in Path("/etc/os-release").read_text(encoding="utf-8").splitlines():
                    if "=" in _line:
                        _key, _value = _line.split("=", 1)
                        _os_release[_key] = _value.strip().strip(chr(34))
                _codename = _os_release.get("VERSION_CODENAME")
                if not _codename:
                    raise RuntimeError("Could not determine the Ubuntu release codename")
                _architecture = subprocess.check_output(
                    ["dpkg", "--print-architecture"], text=True
                ).strip()
                _pgdg_source.write_text(
                    "Types: deb\n"
                    "URIs: https://apt.postgresql.org/pub/repos/apt\n"
                    f"Suites: {_codename}-pgdg\n"
                    f"Architectures: {_architecture}\n"
                    "Components: main\n"
                    f"Signed-By: {_pgdg_key}\n",
                    encoding="utf-8",
                )
                subprocess.check_call(["apt-get", "update", "-qq"], env=_apt_env)
                subprocess.check_call(
                    ["apt-get", "install", "-y", "-qq", "postgresql-client-17"],
                    env=_apt_env,
                )
            except (OSError, subprocess.CalledProcessError) as exc:
                raise RuntimeError(
                    "Could not install PostgreSQL 17 from the official PGDG repository. "
                    "Retry after reconnecting or upload traffic_data_raw.csv instead."
                ) from exc

        if not _pg17_binary.is_file() or not os.access(_pg17_binary, os.X_OK):
            raise RuntimeError(
                f"PostgreSQL 17 installation did not provide an executable: {_pg17_binary}. "
                "Upload traffic_data_raw.csv instead."
            )
        PG_RESTORE_PATH = str(_pg17_binary)

    if PG_RESTORE_PATH is None:
        raise RuntimeError(
            "No pg_restore executable is available. Install PostgreSQL 17 or upload "
            "traffic_data_raw.csv instead."
        )
    _pg_restore_version = subprocess.check_output(
        [PG_RESTORE_PATH, "--version"], text=True
    ).strip()
    print(f"✅ Backup reader ready: {_pg_restore_version} | {PG_RESTORE_PATH}")

In [ ]:
# Cell 2A/2B — Explicit seed or HITL ingestion
# Change only TRAINING_MODE and the corresponding typed source list.

RAW_CSV_PATH = _RAW_DIR / "traffic_data_raw.csv"
BACKUP_PATH = _RAW_DIR / "traffic_data.backup"
DATASET_PACKAGE_PATH = _RAW_DIR / "vaaet-training-dataset-v1.zip"
SEED_PACKAGE_PATH = _DATA_DIR / "vaaet-seed-bootstrap-v1.zip"
training_db_settings = get_optional_database_settings(DatabaseProfile.TRAINING)
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    RAW_SOURCES = [
        PostgresBackupSource(BACKUP_PATH, Path(PG_RESTORE_PATH) if PG_RESTORE_PATH else None),
        # RawCsvSource(RAW_CSV_PATH),
        # PostgresSource(training_db_settings),
    ]
    SEED_SOURCES = []
    FEEDBACK_SOURCES = []
else:
    RAW_SOURCES = []  # standard HITL flow reuses processed features
    SEED_SOURCES = [
        SeedDatasetPackageSource(SEED_PACKAGE_PATH),
    ]
    FEEDBACK_SOURCES = [
        # PostgresSource(training_db_settings),  # effective human labels only
        DatasetPackageSource(DATASET_PACKAGE_PATH),
    ]

TRAINING_INPUTS = TrainingIngestionPlan(
    mode=TRAINING_MODE,
    raw_sources=tuple(RAW_SOURCES),
    seed_sources=tuple(SEED_SOURCES),
    feedback_sources=tuple(FEEDBACK_SOURCES),
    feedback_policy=FeedbackPolicy.VALIDATED_ONLY,
)
training_inputs = load_training_inputs(TRAINING_INPUTS)
df_raw = training_inputs.raw
seed_feature_frame = training_inputs.seed_features
validated_feedback = training_inputs.validated_feedback
confirmed_incidents = training_inputs.confirmed_incidents
DATA_SOURCE = ','.join(training_inputs.provenance['source_type'].astype(str))
display(training_inputs.provenance)
for _source in training_inputs.provenance.to_dict(orient='records'):
    if _source.get('archive_table'):
        print(
            f"Detected backup table: {_source['archive_table']} "
            f"({_source['backup_layout']} raw telemetry) | "
            f"reader={_source['reader_version']} | imported rows={_source['rows']}"
        )
print(f"Mode: {TRAINING_MODE.value}")
print(f"Raw rows: {len(df_raw)} | processed seed: {len(seed_feature_frame)} | validated stable feedback: {len(validated_feedback)} | confirmed incidents: {len(confirmed_incidents)}")
if not df_raw.empty:
    print(f"Raw time range: {df_raw['record_time'].min()} → {df_raw['record_time'].max()}")


In [ ]:
# Cell 2b — Synthetic Data Augmentation
#
# The Belgrano Bridge dataset (Apr–Jul 2025) has no human-confirmed
# Accident and only limited proxy support for Congested.
# We inject physically plausible synthetic sequences so the classifier
# can stress Congested and possible-incident boundaries. Synthetic IDs start at 50001.
# fall before the real data range (2025-04-21…27).

if "df_raw" not in globals() or not isinstance(df_raw, pd.DataFrame):
    raise RuntimeError("Run Cell 2 before synthetic augmentation.")
_n_before = len(df_raw)
if TRAINING_MODE is TrainingMode.HITL_RETRAINING or df_raw.empty:
    _n_synthetic = 0
    print("ℹ️ Synthetic raw augmentation skipped outside the one-time seed bootstrap.")
else:
    df_raw = augment_with_synthetic(
        df_raw, n_accident_seq=10, n_congestion_seq=10, records_per_seq=10, seed=RANDOM_SEED
    )
    _n_synthetic = len(df_raw) - _n_before
    print(f"   Canonical timestamp timezone: {df_raw['record_time'].dt.tz}")

print(f"✅ Synthetic augmentation: {_n_synthetic} records added")
print(f"   Accident sequences: 10 × 10 = 100 records")
print(f"   Congestion sequences: 10 × 10 = 100 records")
print(f"   Total dataset: {len(df_raw)} records ({_n_before} real + {_n_synthetic} synthetic)")
print(f"   Synthetic IDs ≥ 50001 (distinguishable from real data)")

if not df_raw.empty:
    origin_summary = summarize_data_origin(df_raw)
    print("\n📋 Dataset provenance:")
    display(origin_summary) if "display" in dir() else print(origin_summary.to_string(index=False))


## Feature Engineering — From Raw Telemetry to 19 Features

Raw telemetry (speed + counts) does not capture relationships between consecutive records or temporal patterns. Feature engineering produces the 19 canonical variables the model can exploit:

| Feature | Origin | Domain Justification |
|---|---|---|
| `avg_speed` | Direct | Primary indicator of vehicular flow |
| `total_vehicles` | Direct | Absolute traffic volume |
| `count_car` ... `count_bicycle` | Direct (5) | Vehicle composition — trucks and buses impact flow differently than cars |
| `heavy_vehicle_ratio` | Derived | Heavy vehicle proportion — heavy traffic degrades flow more |
| `delta_speed` | Derived (diff) | Acceleration/deceleration between consecutive minutes |
| `delta_count` | Derived (diff) | Volume change rate — detects accumulation |
| `transition_flag` | Derived | Binary signal: simultaneous sharp changes in speed (>8 km/h) and volume (>3 vehicles) |
| `speed_variance` | Derived (rolling) | Recent variability — unstable vs stable traffic |
| `cumulative_delta_speed` | Derived | Accumulated speed trend within the sequence |
| `low_speed_persistence` | Derived | Duration of sustained low-speed conditions |
| `speed_measurement_quality` | Derived | Reliability of the underlying speed estimate |
| `near_zero_motion_ratio` | Direct/derived | Share of tracks with near-zero motion |
| `stationary_confirmed_ratio` | Direct/derived | Share of tracks confirmed as stationary |
| `hour_of_day` | Temporal | Circadian traffic patterns (rush hour, nighttime) |
| `weather_condition` | Simulated | Environmental condition proxy based on hour (nighttime=risk) |

Derived features (`delta_*`, `speed_variance`) introduce NaN in the first records, which are dropped.

> **Nota**: Feature engineering procesa los escenarios sintéticos con la misma semántica temporal. Accident se separa antes del target; Congested sintético sólo puede entrar en train con peso reducido.

In [ ]:
# Cell 3 — Feature Engineering
#
# Uses vaaet.features.engineering.engineer_features() and vaaet.settings.FEATURE_COLS
# (imported in Cell 1). No inline duplication.

audit_frame = (
    (df_raw if not df_raw.empty else seed_feature_frame)
    if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP
    else validated_feedback
)
dataset_audit = audit_training_dataset(audit_frame, require_production_eligible=False)
print("📋 Pre-training dataset audit:")
print(json.dumps(dataset_audit.report, indent=2, default=str))

engineered_proxy_frame = engineer_features(df_raw) if not df_raw.empty else validated_feedback.head(0).copy()
legacy_missing = [column for column in FEATURE_COLS if not engineered_proxy_frame.empty and engineered_proxy_frame[column].isna().any()]
if legacy_missing:
    print("⚠️ Legacy schema v1 rows contain unknown quality evidence.")
    print("   This run may create an EXPERIMENTAL baseline, never a production-approved bundle.")
    print(f"   Conservative zero-fill for baseline-only columns: {legacy_missing}")
    engineered_proxy_frame[legacy_missing] = engineered_proxy_frame[legacy_missing].fillna(0.0)

# Save features CSV for reproducibility (NOT to be used as raw data fallback)
csv_path = os.path.join(_DATA_DIR, "traffic_telemetry.csv")
engineered_proxy_frame.to_csv(csv_path, index=False)

print(f"✅ Raw features engineered: {engineered_proxy_frame.shape[0]} records × {engineered_proxy_frame.shape[1]} columns")
print(f"   Features CSV saved → {os.path.abspath(csv_path)}")
print(f"\n📊 Correlation with avg_speed:")
if not engineered_proxy_frame.empty:
    corr = engineered_proxy_frame[FEATURE_COLS].corr()["avg_speed"].drop("avg_speed").sort_values()
    print(corr.to_string())
else:
    print("Feedback-only run: correlation of raw engineered rows is unavailable.")

## Etiquetado provisional de los tres estados estables

Without manual annotation of thousands of records, we use engineering rules as a ground truth proxy. Thresholds are **calibrated to the Belgrano Bridge real data distribution** (P25 speed ≈ 7.78 km/h, median vehicles ≈ 3, P75 vehicles ≈ 6):

- **Accident (3)** no es una salida del MLP. Los escenarios sintéticos de incidente se reservan para pruebas técnicas del detector jerárquico.
Los valores concretos no se duplican en Markdown: la siguiente celda imprime la matriz vigente directamente desde `LABELING_THRESHOLDS`, única fuente de verdad.

Los datos sintéticos de Congested sólo pueden aumentar `train`; nunca forman parte de validation/test. Estas reglas son etiquetas proxy y no sustituyen ground truth humano.

**Known limitation**: proxy labels are not human ground truth. Effective labels come only from append-only `vaaet_feedback.human_validations`; unreviewed predictions are never supervised targets. See [bias and limitations](../../docs/ml/bias-and-limitations.md).

In [ ]:
# Cell 4 — Auto-Labeling + Class Distribution
#
# Uses the stable three-class proxy labeler. Accident is never an MLP target.
# (imported in Cell 1). No inline duplication.

print("📐 Active weak-label matrix:")
print(json.dumps(dict(LABELING_THRESHOLDS), indent=2))
scenario = engineered_proxy_frame.get("synthetic_scenario", pd.Series("observed", index=engineered_proxy_frame.index))
incident_stress_frame = engineered_proxy_frame.loc[scenario.eq("accident")].copy()
new_proxy_features = engineered_proxy_frame.loc[~scenario.eq("accident")].copy()
if not new_proxy_features.empty:
    new_proxy_features["traffic_state"] = assign_stable_traffic_state(new_proxy_features)
    new_proxy_features["is_human_validated"] = False
    new_proxy_features["feature_schema_version"] = FEATURE_SCHEMA_VERSION
proxy_frames = [frame for frame in (seed_feature_frame, new_proxy_features) if not frame.empty]
proxy_features = pd.concat(proxy_frames, ignore_index=True) if proxy_frames else validated_feedback.head(0).copy()
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP and not new_proxy_features.empty:
    metadata_columns = [column for column in new_proxy_features if column not in FEATURE_COLS]
    seed_export = new_proxy_features[[*metadata_columns, *FEATURE_COLS]]
    create_dataset_package(
        SEED_PACKAGE_PATH, features=seed_export,
        provenance={"training_mode": TRAINING_MODE.value, "supervision": "weak-proxy", "git_commit": GIT_COMMIT},
    )
    print(f"💾 Reusable processed seed package → {SEED_PACKAGE_PATH}")
df_features = compose_supervised_dataset(proxy_features, validated_feedback)
if not validated_feedback.empty:
    print(f"✅ Added {len(validated_feedback)} human-validated stable labels; reserved {len(confirmed_incidents)} confirmed incidents.")

# Distribution
dist = df_features["traffic_state"].value_counts().sort_index()
print("📊 Traffic state distribution:")
for code, count in dist.items():
    pct = 100 * count / len(df_features)
    print(f"   {STATE_LABELS[code]:>10} ({code}): {count:>5} records ({pct:.1f}%)")

# Verify at least 2 classes exist
n_classes = dist.index.nunique()
if n_classes < 2:
    print("🔴 Only 1 class found. Thresholds do not discriminate in this dataset.")
else:
    print(f"\n✅ {n_classes} classes detected")

# Classes without samples
for code, label in {code: STATE_LABELS[code] for code in range(N_MODEL_STATES)}.items():
    if code not in dist.index:
        print(f"⚠️  Class '{label}' ({code}) has no samples — will be excluded from training")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
bars = ax.bar(
    [STATE_LABELS[c] for c in sorted(dist.index)],
    [dist[c] for c in sorted(dist.index)],
    color=[colors[c] for c in sorted(dist.index)],
)
ax.set_ylabel("Records")
ax.set_title("Traffic State Distribution (Auto-Labeling)")
for bar, count in zip(bars, [dist[c] for c in sorted(dist.index)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

support_summary = summarize_state_balance(df_features)
print("\n📋 Support by origin:")
display(support_summary) if "display" in dir() else print(support_summary.to_string(index=False))

print("\n📝 Support notes:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")


## Ground truth humano opcional

Las correcciones humanas se cargan sólo aquí. `Accident` confirmado se reserva para evaluar el detector de incidentes y nunca se convierte en target del MLP de tres clases.


In [ ]:
# Cell 4b — Explicit HITL ingestion report
print(f"Validated stable labels included: {len(validated_feedback)}")
print(f"Confirmed incidents reserved from MLP target: {len(confirmed_incidents)}")
if validated_feedback.empty:
    print("ℹ️ No human feedback source was declared; production holdout gates will remain blocked.")


## Partición temporal, grupos y balanceo conservador

El clasificador aprende únicamente Normal, Reduced y Congested. Accident se excluye del target y se gestiona con la política jerárquica.

**Strategy**:
1. **StandardScaler**: Normalizes features to mean=0, std=1 (required for neural networks)
2. **Train/validation/test**: grupos de clips sin solapamiento y test temporal final
3. **Class weights limitados**: alternativa base; los sintéticos sólo aparecen en train y reciben menor peso efectivo

The scaler is exported as an artifact (`feature_scaler.joblib`) so that production inference uses the same transformation.

In [ ]:
# Cell 5 — Temporal Group Split + Conservative Class Weighting

human_holdout_snapshot = None
if HUMAN_HOLDOUT_FROZEN:
    human_holdout_snapshot = resolve_human_holdout(
        validated_feedback,
        HumanHoldoutConfig(
            store_root=HUMAN_HOLDOUT_STORE_ROOT,
            action=HUMAN_HOLDOUT_ACTION,
            update_reason=HUMAN_HOLDOUT_UPDATE_REASON,
            validation_size=0.2,
            test_size=0.2,
            random_state=RANDOM_SEED,
            git_commit=GIT_COMMIT,
            vaaet_version=package_version("vaaet-ml"),
        ),
    )
    print(f"🔒 Frozen human holdout: {json.dumps(human_holdout_snapshot.descriptor)}")
    HUMAN_HOLDOUT_ACTION = HumanHoldoutAction.REUSE_OR_CREATE
    HUMAN_HOLDOUT_UPDATE_REASON = None

partitions = build_training_partitions(
    proxy_features, validated_feedback, TRAINING_MODE,
    test_size=0.2, validation_size=0.2, random_state=RANDOM_SEED,
    frozen_holdout=human_holdout_snapshot,
)
train_frame = partitions.train
validation_frame = partitions.validation
test_frame = partitions.test

supervision_weight, supervision_report = build_supervision_weights(train_frame, TRAINING_MODE)
active_supervision = supervision_weight > 0
discarded_proxy_rows = int((~active_supervision).sum())
if discarded_proxy_rows:
    train_frame = train_frame.loc[active_supervision].copy()
    supervision_weight = supervision_weight[active_supervision]
    print(f"ℹ️ Removed {discarded_proxy_rows} expired proxy-memory rows before scaling.")
has_effective_proxy_memory = bool(((~train_frame["is_human_validated"].fillna(False)) & pd.Series(supervision_weight > 0, index=train_frame.index)).any())
MODEL_INPUT_POLICY = (
    ModelInputPolicy.LEGACY_V1_BOOTSTRAP
    if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP or has_effective_proxy_memory
    else ModelInputPolicy.CANONICAL_V2
)
X_train_raw = apply_model_input_policy(train_frame, MODEL_INPUT_POLICY).to_numpy()
X_validation_raw = apply_model_input_policy(validation_frame, MODEL_INPUT_POLICY).to_numpy()
X_test_raw = apply_model_input_policy(test_frame, MODEL_INPUT_POLICY).to_numpy()
y_train = train_frame["traffic_state"].to_numpy(dtype=int)
y_validation = validation_frame["traffic_state"].to_numpy(dtype=int)
y_test = test_frame["traffic_state"].to_numpy(dtype=int)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_validation = scaler.transform(X_validation_raw)
X_test = scaler.transform(X_test_raw)

balance_candidates = build_balance_candidates(
    train_frame, supervision_weight, random_state=RANDOM_SEED
)
synthetic_train = train_frame.get("data_origin", pd.Series("real", index=train_frame.index)).eq("synthetic").to_numpy()

print("📊 Leakage-aware three-way partition:")
print(f"   Train={len(train_frame)} | Validation={len(validation_frame)} | Test={len(test_frame)}")
print(f"   Balance candidates: {[strategy.value for strategy in balance_candidates]}")
print(f"   Input policy: {MODEL_INPUT_POLICY.value}")
print(f"   Supervision policy: {json.dumps(supervision_report, default=str)}")
print(f"   Synthetic rows in validation/test: {(validation_frame.get('data_origin') == 'synthetic').sum() if 'data_origin' in validation_frame else 0}/{(test_frame.get('data_origin') == 'synthetic').sum() if 'data_origin' in test_frame else 0}")
partition_summary = pd.DataFrame([
    {"partition": name, "records": len(frame), "clips": build_group_ids(frame).nunique(), "start": normalize_timestamp_series(frame["record_time"]).min(), "end": normalize_timestamp_series(frame["record_time"]).max(), "missing_features": int(frame[FEATURE_COLS].isna().sum().sum())}
    for name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame))
])
display(partition_summary) if "display" in dir() else print(partition_summary.to_string(index=False))

# Export scaler
scaler_path = os.path.join(_MODEL_DIR, "feature_scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"\n💾 Scaler saved → {os.path.abspath(scaler_path)}")

print("✅ No default SMOTE: validation and test remain real and untouched.")


## Model Architecture — Tabular MLP

The model is a **Multi-Layer Perceptron (MLP)** implemented with `tf.keras.Sequential`. The architecture is deliberately simple — designed to validate the complete pipeline. A future iteration may evolve to LSTM with temporal memory.

**Why these dimensions?**
- **Dense(64)**: Input layer with sufficient capacity to learn non-linear combinations of 19 features
- **Dense(32)**: Compression layer that forces more abstract representations
- **BatchNormalization**: Stabilizes and accelerates training by normalizing activations between layers
- **Dropout(0.3 → 0.2)**: Decreasing regularization — more aggressive near the input (where there is more redundancy)
- **Dense(3, softmax)**: distribución sobre Normal, Reduced y Congested; no produce Accident

In [ ]:
# Cell 6 — Model Definition + Training

n_classes: int = N_MODEL_STATES
n_features: int = X_train.shape[1]

print(f"🏗️ Building MLP model: {n_features} features → {n_classes} classes")
print("   Outputs: Normal / Reduced / Congested (Accident is not learned)")

def build_mlp_model() -> Sequential:
    candidate_model = Sequential([
        Input(shape=(n_features,)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(n_classes, activation="softmax"),
    ], name="traffic_state_classifier")
    candidate_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return candidate_model

model = build_mlp_model()

model.summary()

# Train and compare all conservative balance alternatives on validation only.
print("\n🚀 Comparing conservative balance strategies...")
_training_engine = get_engine(training_db_settings) if training_db_settings is not None else None
_training_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.TRAINING, git_commit=GIT_COMMIT, source_kind="composed-dataset", input_rows=len(X_train), model_version=MODEL_VERSION)
try:
    with pipeline_run(_training_metadata, engine=_training_engine, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs") as _run:
        _candidate_results = []
        _trained_candidates = {}
        for _strategy, _candidate in balance_candidates.items():
            tf.keras.backend.clear_session()
            tf.random.set_seed(RANDOM_SEED)
            _positions = _candidate.row_positions
            _candidate_frame = train_frame.iloc[_positions].copy()
            _candidate_y = y_train[_positions]
            _candidate_weights, _class_weights = compute_capped_balanced_weights(
                _candidate_y, _candidate.supervision_weights
            )
            if _strategy is BalanceStrategy.SYNTHETIC_CONGESTION:
                _candidate_weights = cap_synthetic_congested_weight(
                    _candidate_frame, _candidate_weights
                )
            _candidate_model = build_mlp_model()
            _candidate_history = _candidate_model.fit(
                X_train[_positions],
                _candidate_y,
                epochs=120,
                batch_size=32,
                validation_data=(X_validation, y_validation),
                sample_weight=_candidate_weights,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
                    ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5, min_lr=1e-6, verbose=0),
                ],
                verbose=0,
            )
            _raw_validation_proba = _candidate_model.predict(X_validation, verbose=0)
            _temperature = fit_temperature(_raw_validation_proba, y_validation)
            _validation_proba = apply_temperature_scaling(_raw_validation_proba, _temperature)
            _policy = select_validation_decision_policy(
                validation_frame, y_validation, _validation_proba, temperature=_temperature
            )
            _classified = classify_telemetry_dataframe(
                validation_frame, _candidate_model, scaler,
                decision_policy=_policy, input_policy=MODEL_INPUT_POLICY,
            )
            _predicted = _classified["traffic_state"].to_numpy(dtype=int)
            _cost = expected_confusion_cost(y_validation, _predicted)
            _false_congested = float(((_predicted == 2) & (y_validation != 2)).mean())
            _f1 = f1_score(
                y_validation, _predicted, labels=[0, 1, 2],
                average="macro", zero_division=0,
            )
            _selection_score = _cost + 4.0 * _false_congested
            _candidate_results.append({
                "strategy": _strategy.value,
                "training_rows": len(_positions),
                "validation_f1_macro": _f1,
                "validation_confusion_cost": _cost,
                "validation_false_congested_rate": _false_congested,
                "selection_score": _selection_score,
            })
            _trained_candidates[_strategy] = (
                _candidate_model, _candidate_history, _candidate_weights, _class_weights
            )
        balance_selection_report = pd.DataFrame(_candidate_results).sort_values(
            ["selection_score", "validation_false_congested_rate", "validation_f1_macro"],
            ascending=[True, True, False],
        ).reset_index(drop=True)
        SELECTED_BALANCE_STRATEGY = BalanceStrategy(balance_selection_report.iloc[0]["strategy"])
        model, history, sample_weight, class_weights = _trained_candidates[SELECTED_BALANCE_STRATEGY]
        _run.set_output_rows(len(balance_candidates[SELECTED_BALANCE_STRATEGY].row_positions))
    TRAINING_PIPELINE_RUN_ID = str(_run.id)
finally:
    if _training_engine is not None:
        _training_engine.dispose()

display(balance_selection_report) if "display" in dir() else print(balance_selection_report.to_string(index=False))
print(f"✅ Selected balance strategy: {SELECTED_BALANCE_STRATEGY.value}")
model.summary()

# Training visualization for the selected candidate.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history["loss"], label="Train Loss")
ax1.plot(history.history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss During Training")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history["accuracy"], label="Train Accuracy")
ax2.plot(history.history["val_accuracy"], label="Val Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy During Training")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n✅ Training completed — best epoch: {best_epoch}")

## Evaluation — Classification Metrics

The key metrics for this classifier are:

- **F1-macro de tres estados** ≥ 0.88, siempre acompañado por soporte real y número de clips
- **Coste de confusión**: penaliza especialmente los errores directos Normal ↔ Congested
- **ECE**: impide presentar softmax como probabilidad fiable sin comprobar calibración
- **Confusion matrix**: Identifies systematic confusions (e.g., Normal↔Reduced is the most likely confusion pair)

Accident no tiene recall publicable sin casos reales. El bundle se marca experimental mientras falte telemetría v2 y holdout humano.

In [ ]:
# Cell 7 — Evaluation + Model Export

# Calibrate and select thresholds on validation only.
validation_proba_raw = model.predict(X_validation, verbose=0)
temperature = fit_temperature(validation_proba_raw, y_validation)
validation_proba = apply_temperature_scaling(validation_proba_raw, temperature)
decision_policy = select_validation_decision_policy(validation_frame, y_validation, validation_proba, temperature=temperature)
print(f"🎛️ Validation-selected decision policy: {decision_policy}")

# Evaluate the exact production chain on the frozen real test groups.
y_proba_raw = model.predict(X_test, verbose=0)
y_proba = apply_temperature_scaling(y_proba_raw, temperature)
classified_test = classify_telemetry_dataframe(test_frame, model, scaler, decision_policy=decision_policy, input_policy=MODEL_INPUT_POLICY)
y_model_pred = y_proba.argmax(axis=1).astype(int)
y_pred = classified_test["traffic_state"].to_numpy(dtype=int)
direct_target_accuracy = float((y_model_pred == y_test).mean())
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    weak_supervision_fidelity = direct_target_accuracy
    human_holdout_direct_accuracy = None
    print(f"Weak-supervision fidelity (direct MLP): {direct_target_accuracy:.2%}")
else:
    weak_supervision_fidelity = None
    human_holdout_direct_accuracy = direct_target_accuracy
    print(f"Human-holdout direct MLP accuracy: {direct_target_accuracy:.2%}")

# Present class names
present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [STATE_LABELS[c] for c in present_classes]

# Classification Report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(
    y_test, y_pred,
    labels=present_classes,
    target_names=target_names,
    zero_division=0,
)
print(report)
support_with_intervals = build_classification_support_table(y_test, y_pred)
print("\nPer-class support and 95% Wilson intervals:")
display(support_with_intervals) if "display" in dir() else print(support_with_intervals.to_string(index=False))

# F1-macro
f1_macro = f1_score(y_test, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)
confusion_cost = expected_confusion_cost(y_test, y_pred)
ece = expected_calibration_error(y_test, y_proba)
brier = multiclass_brier_score(y_test, y_proba)
direct_extreme_error = float((((y_test == 0) & (y_pred == 2)) | ((y_test == 2) & (y_pred == 0))).mean())
automatic_accident_states = int(classified_test["traffic_state"].eq(3).sum())
incident_candidate_count = int(classified_test["accident_alert_started"].sum())
reliable_negative_mask = classified_test["measurement_reliable"].fillna(False).astype(bool)
negative_exposure_hours = float(reliable_negative_mask.sum()) / 60.0
reliable_incident_candidates = int((classified_test["accident_alert_started"] & reliable_negative_mask).sum())
false_candidates_per_hour = reliable_incident_candidates / negative_exposure_hours if negative_exposure_hours else float("nan")
if automatic_accident_states != 0:
    raise RuntimeError("Invariant violated: Accident was produced automatically.")
synthetic_incident_episode_sensitivity = None
if not incident_stress_frame.empty:
    stress = incident_stress_frame.copy()
    stress["traffic_state"] = 2
    stress["state_label"] = STATE_LABELS[2]
    stress["confidence"] = 0.5
    stress = apply_conservative_accident_gate(stress)
    episode_hits = stress.groupby("clip_id")["accident_alert_started"].any()
    synthetic_incident_episode_sensitivity = float(episode_hits.mean())
    print(f"Synthetic incident stress sensitivity: {synthetic_incident_episode_sensitivity:.2%} (technical only; not real recall)")
confirmed_incident_candidate_sensitivity = None
confirmed_incident_support = len(confirmed_incidents)
if confirmed_incident_support:
    human_incident_context = pd.concat([validated_feedback, confirmed_incidents], ignore_index=True).sort_values(["clip_id", "record_time"])
    human_incident_context["traffic_state"] = 2
    human_incident_context["state_label"] = STATE_LABELS[2]
    human_incident_context["confidence"] = 0.5
    human_incident_context = apply_conservative_accident_gate(human_incident_context)
    confirmed_keys = set(zip(confirmed_incidents["clip_id"], normalize_timestamp_series(confirmed_incidents["record_time"])))
    evaluated_keys = list(zip(human_incident_context["clip_id"], normalize_timestamp_series(human_incident_context["record_time"])))
    confirmed_mask = pd.Series([key in confirmed_keys for key in evaluated_keys], index=human_incident_context.index)
    confirmed_incident_candidate_sensitivity = float(human_incident_context.loc[confirmed_mask, "accident_rule_triggered"].mean())
    print(f"Confirmed-incident candidate detection: {confirmed_incident_candidate_sensitivity:.2%} (support={confirmed_incident_support}; not operational recall until support/diversity are sufficient)")
else:
    print("Confirmed-incident candidate detection: unsupported (0 human-confirmed incidents)")
print(f"{'F1-macro':>15}: {f1_macro:.4f}")
print(f"{'Expected cost':>15}: {confusion_cost:.4f}")
print(f"{'ECE':>15}: {ece:.4f}")
print(f"{'Brier':>15}: {brier:.4f}")
print(f"{'Normal↔Congested':>15}: {direct_extreme_error:.2%}")
print(f"{'Incident candidates/hour':>15}: {false_candidates_per_hour:.6f} over {negative_exposure_hours:.2f} h")
if negative_exposure_hours < 300:
    print("⚠️ Incident false-alert rate is preliminary; approximately 300 negative hours are required.")

if f1_macro >= 0.88:
    print(f"✅ F1-macro MEETS the target (≥ 0.88)")
else:
    print(f"⚠️  F1-macro BELOW target (≥ 0.88) — bundle remains experimental")

# Confusion Matrix
model_cm = confusion_matrix(y_test, y_model_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(model_cm, annot=True, fmt="d", cmap="Greens", xticklabels=[STATE_LABELS[c] for c in range(3)], yticklabels=[STATE_LABELS[c] for c in range(3)], ax=ax)
ax.set(xlabel="Direct MLP prediction", ylabel="Proxy/human target", title="Direct MLP confusion matrix")
plt.tight_layout()
plt.show()
cm = confusion_matrix(y_test, y_pred, labels=present_classes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — F1-macro: {f1_macro:.4f}")
plt.tight_layout()
plt.show()

# Reliability diagram for calibrated top-label confidence.
confidence = y_proba.max(axis=1)
correct = y_proba.argmax(axis=1).eq(y_test) if isinstance(y_test, pd.Series) else y_proba.argmax(axis=1) == y_test
edges = np.linspace(0.0, 1.0, 11)
bin_confidence, bin_accuracy = [], []
for lower, upper in zip(edges[:-1], edges[1:]):
    mask = (confidence > lower) & (confidence <= upper)
    if mask.any():
        bin_confidence.append(float(confidence[mask].mean()))
        bin_accuracy.append(float(correct[mask].mean()))
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Ideal")
ax.plot(bin_confidence, bin_accuracy, marker="o", label="Calibrated MLP")
ax.set(xlabel="Mean confidence", ylabel="Observed accuracy", title="Reliability diagram")
ax.legend()
plt.show()

# Per-class recall
print("\n📊 Per-class recall:")
for i, cls in enumerate(present_classes):
    row_sum = cm[i].sum()
    recall = cm[i, i] / row_sum if row_sum > 0 else 0.0
    status = "✅" if recall > 0 else "🔴"
    print(f"   {status} {STATE_LABELS[cls]:>10}: {recall:.4f}")

# Export model
model_path = os.path.join(_MODEL_DIR, "traffic_classifier.keras")
model.save(model_path)

# Export the canonical mapping required by the portable serving contract.
label_mapping = dict(STATE_LABELS)
label_path = os.path.join(_MODEL_DIR, "label_mapping.joblib")
joblib.dump(label_mapping, label_path)
human_test_only = bool("is_human_validated" in test_frame and test_frame["is_human_validated"].fillna(False).all())
human_holdout = bool(human_test_only and human_holdout_snapshot is not None)
promotion_blockers = list(dataset_audit.blockers)
if TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    promotion_blockers.append("seed bootstrap uses weak proxy supervision and is pilot-only")
if not human_holdout:
    promotion_blockers.append("validation/test are not a frozen human-validated holdout")
metric_report = classification_report(y_test, y_pred, labels=[0, 1, 2], output_dict=True, zero_division=0)
metric_gates = {
    "f1_macro": f1_macro >= 0.88,
    "normal_precision": metric_report["0"]["precision"] >= 0.93,
    "normal_recall": metric_report["0"]["recall"] >= 0.93,
    "reduced_precision": metric_report["1"]["precision"] >= 0.88,
    "reduced_recall": metric_report["1"]["recall"] >= 0.90,
    "congested_precision": metric_report["2"]["precision"] >= 0.90,
    "congested_recall": metric_report["2"]["recall"] >= 0.85,
    "direct_normal_congested_error": direct_extreme_error <= 0.01,
    "ece": ece <= 0.05,
}
for gate_name, passed in metric_gates.items():
    if not passed:
        promotion_blockers.append(f"metric gate failed: {gate_name}")
congested_test = test_frame.loc[test_frame["traffic_state"].eq(2)]
congested_clips = build_group_ids(congested_test).nunique() if not congested_test.empty else 0
if len(congested_test) < 100 or congested_clips < 20:
    promotion_blockers.append(f"Congested support insufficient: {len(congested_test)} minutes / {congested_clips} clips")
if negative_exposure_hours < 300:
    promotion_blockers.append(f"incident negative exposure insufficient: {negative_exposure_hours:.2f}/300 h")
elif false_candidates_per_hour >= 0.01:
    promotion_blockers.append("incident candidate rate is not below 1 per 100 hours")
production_eligible = not promotion_blockers and TRAINING_MODE is TrainingMode.HITL_RETRAINING
training_lifecycle = build_training_lifecycle(TRAINING_MODE, MODEL_INPUT_POLICY, production_eligible=production_eligible)
create_manifest(
    _MODEL_DIR,
    metrics={"f1_macro": float(f1_macro), "weak_supervision_fidelity": weak_supervision_fidelity, "human_holdout_direct_accuracy": human_holdout_direct_accuracy, "expected_confusion_cost": confusion_cost, "ece": ece, "brier_score": brier, "direct_normal_congested_error": direct_extreme_error, "automatic_accident_states": automatic_accident_states, "incident_candidate_count": reliable_incident_candidates, "negative_exposure_hours": negative_exposure_hours, "false_candidates_per_hour": false_candidates_per_hour, "synthetic_incident_episode_sensitivity": synthetic_incident_episode_sensitivity, "confirmed_incident_support": confirmed_incident_support, "confirmed_incident_candidate_sensitivity": confirmed_incident_candidate_sensitivity, "selected_balance_strategy": SELECTED_BALANCE_STRATEGY.value, "balance_validation_candidates": balance_selection_report.to_dict(orient="records"), "production_eligible": production_eligible},
    data_provenance={
        "origin": "training-notebook",
        "dataset_source": str(DATA_SOURCE),
        "record_count": int(len(df_features)),
        "real_record_count_before_engineering": int(_n_before),
        "synthetic_record_count_before_engineering": int(_n_synthetic),
        "synthetic_data_included": bool(synthetic_train.any()),
        "synthetic_records_in_validation": 0,
        "synthetic_records_in_test": 0,
        "telemetry_v2_coverage": dataset_audit.report["telemetry_v2_coverage"],
        "human_test_only": human_test_only,
        "human_holdout": human_holdout,
        "production_eligible": production_eligible,
        "promotion_blockers": promotion_blockers,
    },
    decision_policy=decision_policy,
    training_lifecycle=training_lifecycle,
    human_holdout=human_holdout_snapshot.descriptor if human_holdout_snapshot is not None else None,
)
print(f"\nDeployment stage: {training_lifecycle['deployment_stage'].upper()}")
for blocker in promotion_blockers:
    print(f"   - {blocker}")

print(f"\n💾 Artifacts exported:")
print(f"   Model  → {os.path.abspath(model_path)} ({os.path.getsize(model_path) / 1024:.1f} KB)")
print(f"   Labels → {os.path.abspath(label_path)} (classes: {list(label_mapping.values())})")
print(f"   Scaler → {os.path.abspath(os.path.join(_MODEL_DIR, 'feature_scaler.joblib'))}")

print("\n📝 Support notes:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")

## K-Fold Cross-Validation (Optional)

Provides a more robust estimate of model generalization by training and evaluating on 5 different data splits. This addresses potential optimistic bias from a single 80/20 split.

- **StratifiedGroupKFold**: mantiene clips completos dentro de cada fold
- El scaler y los class weights se ajustan exclusivamente en el train de cada fold
- Reports mean +/- std of F1-macro across all folds

In [ ]:
# Cell 7b — K-Fold Cross-Validation (Optional)
#
# Run this cell for a more robust generalization estimate.
# It does NOT replace the single-split model (exported in Cell 7).

from sklearn.model_selection import StratifiedGroupKFold

cv_source = validated_feedback if TRAINING_MODE is TrainingMode.HITL_RETRAINING else df_features
cv_frame = cv_source.loc[~cv_source.get("data_origin", pd.Series("real", index=cv_source.index)).eq("synthetic")].copy()
groups_all = build_group_ids(cv_frame).values
N_FOLDS: int = min(5, len(np.unique(groups_all)))
if N_FOLDS < 2:
    raise RuntimeError("At least two real groups are required for grouped cross-validation.")
kf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Only real, unscaled records can enter validation folds.
X_all = apply_model_input_policy(cv_frame, MODEL_INPUT_POLICY).values
y_all = cv_frame["traffic_state"].values

fold_f1_scores: list[float] = []

print(f"🔁 {N_FOLDS}-Fold Stratified Cross-Validation")
print("=" * 50)

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all, y_all, groups_all), 1):
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]

    # Scale per fold
    fold_scaler = StandardScaler()
    X_tr = fold_scaler.fit_transform(X_tr)
    X_val = fold_scaler.transform(X_val)

    fold_classes = np.unique(y_tr)
    fold_balanced = compute_class_weight(class_weight="balanced", classes=fold_classes, y=y_tr)
    fold_weights = {int(code): min(float(weight), 4.0) for code, weight in zip(fold_classes, fold_balanced)}
    fold_sample_weight = np.array([fold_weights[int(code)] for code in y_tr])

    # Build model (identical architecture)
    fold_model = Sequential([
        Input(shape=(X_tr.shape[1],)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(N_MODEL_STATES, activation="softmax"),
    ])
    fold_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    fold_model.fit(
        X_tr, y_tr,
        epochs=200,
        batch_size=32,
        validation_data=(X_val, y_val),
        sample_weight=fold_sample_weight,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
        ],
        verbose=0,
    )

    y_pred_fold = fold_model.predict(X_val, verbose=0).argmax(axis=1).astype(int)

    fold_support = {code: int((y_val == code).sum()) for code in range(N_MODEL_STATES)}
    missing_fold_classes = [STATE_LABELS[code] for code, support in fold_support.items() if support == 0]
    fold_f1 = f1_score(y_val, y_pred_fold, labels=[0, 1, 2], average="macro", zero_division=0)
    fold_f1_scores.append(np.nan if missing_fold_classes else fold_f1)
    status = f"INSUFFICIENT missing={missing_fold_classes}" if missing_fold_classes else f"F1-macro={fold_f1:.4f}"
    print(f"   Fold {fold_idx}: {status} | support={fold_support}")

evaluable_fold_scores = np.asarray(fold_f1_scores, dtype=float)
mean_f1 = np.nanmean(evaluable_fold_scores) if np.isfinite(evaluable_fold_scores).any() else float("nan")
std_f1 = np.nanstd(evaluable_fold_scores) if np.isfinite(evaluable_fold_scores).any() else float("nan")

print("=" * 50)
print(f"📊 K-Fold F1-macro: {mean_f1:.4f} ± {std_f1:.4f}")

if mean_f1 >= 0.88:
    print(f"✅ Cross-validated F1-macro MEETS target (≥ 0.88)")
else:
    print(f"⚠️  Cross-validated F1-macro BELOW target (≥ 0.88)")
    print(f"   Consider adjusting labeling thresholds or model architecture")

In [ ]:
# Cell 7c — Export Artifacts to Google Drive (Colab only)
#
# Copies trained artifacts to Google Drive so the inference workflow can load them
# even after a Colab runtime reset.  Skipped silently when running locally.

import shutil

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)

        _drive_dest = os.path.join("/content/drive", DRIVE_ARTIFACT_DIR)
        os.makedirs(_drive_dest, exist_ok=True)

        _artifact_files = [
            os.path.join(_MODEL_DIR, "traffic_classifier.keras"),
            os.path.join(_MODEL_DIR, "feature_scaler.joblib"),
            os.path.join(_MODEL_DIR, "label_mapping.joblib"),
            os.path.join(_MODEL_DIR, MANIFEST_FILE),
        ]

        _copied = 0
        for src_path in _artifact_files:
            if os.path.isfile(src_path):
                shutil.copy2(src_path, _drive_dest)
                _copied += 1
            else:
                print(f"⚠️  Not found (skipped): {src_path}")

        if _copied == len(_artifact_files):
            print(f"✅ {_copied} artifacts exported to Google Drive:")
            print(f"   {_drive_dest}")
        else:
            print(f"⚠️  Only {_copied}/{len(_artifact_files)} artifacts copied")

        if os.path.isfile(SEED_PACKAGE_PATH):
            _drive_seed_dir = os.path.join("/content/drive", "MyDrive", "vaaet-ml", "data", "processed")
            os.makedirs(_drive_seed_dir, exist_ok=True)
            shutil.copy2(SEED_PACKAGE_PATH, _drive_seed_dir)
            print(f"✅ Reusable seed package exported to {_drive_seed_dir}")
    except Exception as e:
        print(f"⚠️  Drive export skipped: {e}")
        print("   Artifacts are available locally — run inference in the same session")
else:
    print("ℹ️  Local environment — Drive export skipped")
    print(f"   Artifacts at: {os.path.abspath(_MODEL_DIR)}")

## Persistencia operacional

El perfil `training` es estrictamente read-only. Este notebook genera artefactos locales/Drive/DVC y no escribe predicciones operacionales. La persistencia pertenece exclusivamente al workflow de inferencia.

La base operativa se organiza en `vaaet_raw.traffic_data`, `vaaet_ml.telemetry_features`, `vaaet_ml.traffic_predictions` y `vaaet_feedback.human_validations`. Las validaciones humanas son append-only y nunca son modificadas por un reentrenamiento.

In [ ]:
# Cell 8 — Read-only training boundary
print("✅ Training completed without operational database writes.")
print(f"   Artifacts: {os.path.abspath(_MODEL_DIR)}")
print("   Use analyze_traffic_video.ipynb with the inference profile to persist predictions.")
